In [1]:
import numpy as np
import pandas as pd
from benchmark import *

### Python timing

In [2]:
# benchmark_settings = {
#     "model_size": ["small", "medium", "large", "xl", "2.7b"],
#     "d_model": [768, 1024, 1280, 1600, 2560],
#     "d_ff": [3072, 4096, 5120, 10240, 12888],
#     "num_layers": [12, 24, 36, 32, 50],
#     "num_heads": [12, 16, 20, 32, 36]
# }

## Standard deviation with warmup iterations removed is much smaller than the average, so we can report the average as a reliable benchmark.
## The warmup iterations are impacted by factors including CUDA kernel compilation and GPU power ramp-up, so we exclude them from the benchmark.

## modifications applied to fit into RTX 5080
benchmark_settings = {
    "model_size": ["tiny", "small", "medium", "deep-narrow", "ffn-heavy"],
    "d_model": [384, 768, 1024, 512, 768],
    "d_ff": [1536, 3072, 4096, 2048, 6144],
    "num_layers": [6, 12, 24, 20, 12],
    "num_heads": [6, 12, 16, 8, 12]
}

params = ["d_model", "d_ff", "num_layers", "num_heads"]
length = len(benchmark_settings["model_size"])
for key in params:
    assert len(benchmark_settings[key]) == length, f"Length mismatch for {key}"

output_dict = {key: list() for key in list(benchmark_settings.keys()) + ["warmup_iters", "forward_avg", "forward_std", "backward_avg", "backward_std", "optimizer_avg", "optimizer_std"]}
for index in range(length):
    input_dict = {key: benchmark_settings[key][index] for key in params}
    result = benchmark_python(**input_dict)
    for warmup_iters in [0, 1, 2, 5]:
        output_dict["warmup_iters"].append(warmup_iters)
        for key, val in result.items():
            output_dict[f"{key}_avg"].append(np.mean(val[warmup_iters:]))
            output_dict[f"{key}_std"].append(np.std(val[warmup_iters:]))
        for u, v in benchmark_settings.items():
            output_dict[u].append(v[index])
    print(f"Done {benchmark_settings['model_size'][index]}: ", f"peak={torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    torch.cuda.reset_peak_memory_stats()

output_df = pd.DataFrame({**output_dict})
output_df

Done tiny:  peak=1.5GB
Done small:  peak=5.4GB
Done medium:  peak=14.7GB
Done deep-narrow:  peak=5.4GB
Done ffn-heavy:  peak=7.9GB


,model_size,d_model,d_ff,num_layers,num_heads,warmup_iters,forward_avg,forward_std,backward_avg,backward_std,optimizer_avg,optimizer_std
0,tiny,384,1536,6,6,0,24.195517,90.305493,25.894562,33.885657,15.595810,10.173923
1,tiny,384,1536,6,6,1,11.317441,5.410183,21.132172,6.136227,15.515005,10.261318
2,tiny,384,1536,6,6,2,10.896865,4.605708,21.212182,6.174468,15.191809,10.117805
3,tiny,384,1536,6,6,5,10.505145,4.031634,21.147004,6.107509,14.384684,9.606311
4,small,768,3072,12,12,0,38.555465,5.561823,75.691157,3.321290,31.361577,17.149278
5,small,768,3072,12,12,1,37.918600,3.359360,75.647189,3.340573,31.469591,17.306541
6,small,768,3072,12,12,2,37.608853,2.611359,75.639364,3.374747,31.710059,17.404680
7,small,768,3072,12,12,5,37.700511,2.671702,75.772068,3.441629,32.509987,17.686255
8,medium,1024,4096,24,16,0,235.942516,101.390334,415.531400,174.773052,76.437339,22.914455
9,medium,1024,4096,24,16,1,237.897456,101.482541,419.455659,174.353116,76.891506,22.923232


### Nsight timing and mixed precision

In [2]:
## Generate commands to be executed in command lines
## nsys profile -o report python -m cs336_systems.benchmark --d_model 768

## modifications applied to fit into RTX 5080
benchmark_settings = {
    "model_size": ["tiny", "small", "medium", "deep-narrow", "ffn-heavy"],
    "d_model": [384, 768, 1024, 512, 768],
    "d_ff": [1536, 3072, 4096, 2048, 6144],
    "num_layers": [6, 12, 24, 20, 12],
    "num_heads": [6, 12, 16, 8, 12]
}

params = ["d_model", "d_ff", "num_layers", "num_heads"]
length = len(benchmark_settings["model_size"])
for key in params:
    assert len(benchmark_settings[key]) == length, f"Length mismatch for {key}"

print("rm cs336_systems/nsys_reports/*.qdstrm cs336_systems/nsys_reports/*.nsys-rep cs336_systems/nsys_reports/*.sqlite cs336_systems/nsys_reports/*.csv")

for index in range(length):
    model_size = benchmark_settings["model_size"][index]
    input_dict = {key: benchmark_settings[key][index] for key in params}
    command = f"nsys profile -o cs336_systems/nsys_reports/{model_size} python -m cs336_systems.benchmark --d_model {input_dict['d_model']} --d_ff {input_dict['d_ff']} --num_layers {input_dict['num_layers']} --num_heads {input_dict['num_heads']}"
    print(command)

for index in range(length):
    model_size = benchmark_settings["model_size"][index]
    command = f"/usr/lib/nsight-systems/host-linux-x64/QdstrmImporter -i cs336_systems/nsys_reports/{model_size}.qdstrm -o cs336_systems/nsys_reports/{model_size}.nsys-rep"
    print(command)

for index in range(length):
    model_size = benchmark_settings["model_size"][index]
    command = f"nsys stats --format csv -r nvtxsum cs336_systems/nsys_reports/{model_size}.nsys-rep -o cs336_systems/nsys_reports/{model_size}"
    print(command)

rm cs336_systems/nsys_reports/*.qdstrm cs336_systems/nsys_reports/*.nsys-rep cs336_systems/nsys_reports/*.sqlite cs336_systems/nsys_reports/*.csv
nsys profile -o cs336_systems/nsys_reports/tiny python -m cs336_systems.benchmark --d_model 384 --d_ff 1536 --num_layers 6 --num_heads 6
nsys profile -o cs336_systems/nsys_reports/small python -m cs336_systems.benchmark --d_model 768 --d_ff 3072 --num_layers 12 --num_heads 12
nsys profile -o cs336_systems/nsys_reports/medium python -m cs336_systems.benchmark --d_model 1024 --d_ff 4096 --num_layers 24 --num_heads 16
nsys profile -o cs336_systems/nsys_reports/deep-narrow python -m cs336_systems.benchmark --d_model 512 --d_ff 2048 --num_layers 20 --num_heads 8
nsys profile -o cs336_systems/nsys_reports/ffn-heavy python -m cs336_systems.benchmark --d_model 768 --d_ff 6144 --num_layers 12 --num_heads 12
/usr/lib/nsight-systems/host-linux-x64/QdstrmImporter -i cs336_systems/nsys_reports/tiny.qdstrm -o cs336_systems/nsys_reports/tiny.nsys-rep
/usr/l

In [3]:
## NVTX measures only GPU time and thus output values are lower than python timeit output values.
benchmark_settings = {
    "model_size": ["tiny", "small", "medium", "deep-narrow", "ffn-heavy"],
    "d_model": [384, 768, 1024, 512, 768],
    "d_ff": [1536, 3072, 4096, 2048, 6144],
    "num_layers": [6, 12, 24, 20, 12],
    "num_heads": [6, 12, 16, 8, 12]
}

output_dict = {f"{key}_{agg}": list() for key in ["forward", "backward", "optimizer"] for agg in ["avg", "std"]}
for index in range(length):
    model_size = benchmark_settings["model_size"][index]
    result = pd.read_csv(f"./nsys_reports/{model_size}_nvtxsum.csv")
    for key in ["forward", "backward", "optimizer"]:
        output_dict[f"{key}_avg"].append(result[result["Range"] == f"bench/{key}"]["Avg (ns)"].values[0] / 1000000.0)
        output_dict[f"{key}_std"].append(result[result["Range"] == f"bench/{key}"]["StdDev (ns)"].values[0] / 1000000.0)

output_df = pd.DataFrame({**benchmark_settings, **output_dict})
output_df.to_csv("benchmark_timing_nvtx_nomixed.csv", index=False)
output_df

,model_size,d_model,d_ff,num_layers,num_heads,forward_avg,forward_std,backward_avg,backward_std,optimizer_avg,optimizer_std
0,tiny,384,1536,6,6,8.235159,5.550969,15.783632,11.497065,14.532891,14.154729
1,small,768,3072,12,12,13.768401,5.250832,55.725192,11.036021,32.475617,22.543723
2,medium,1024,4096,24,16,54.809770,11.074749,397.147378,32.469451,62.111377,27.533643
3,deep-narrow,512,2048,20,8,25.023518,12.006970,71.924211,30.282646,39.140697,19.665779
4,ffn-heavy,768,6144,12,12,14.946307,4.681391,83.871289,9.496351,29.327223,25.007633


In [2]:
## Generate commands to be executed in command lines
## nsys profile -o report python -m cs336_systems.benchmark --d_model 768

## modifications applied to fit into RTX 5080
benchmark_settings = {
    "model_size": ["tiny", "small", "medium", "deep-narrow", "ffn-heavy"],
    "d_model": [384, 768, 1024, 512, 768],
    "d_ff": [1536, 3072, 4096, 2048, 6144],
    "num_layers": [6, 12, 24, 20, 12],
    "num_heads": [6, 12, 16, 8, 12]
}

params = ["d_model", "d_ff", "num_layers", "num_heads"]
length = len(benchmark_settings["model_size"])
for key in params:
    assert len(benchmark_settings[key]) == length, f"Length mismatch for {key}"

print("rm cs336_systems/nsys_reports/*.qdstrm cs336_systems/nsys_reports/*.nsys-rep cs336_systems/nsys_reports/*.sqlite cs336_systems/nsys_reports/*.csv")

for index in range(length):
    model_size = benchmark_settings["model_size"][index]
    input_dict = {key: benchmark_settings[key][index] for key in params}
    command = f"nsys profile -o cs336_systems/nsys_reports/{model_size} python -m cs336_systems.benchmark --d_model {input_dict['d_model']} --d_ff {input_dict['d_ff']} --num_layers {input_dict['num_layers']} --num_heads {input_dict['num_heads']} --mixed_precision True"
    print(command)

for index in range(length):
    model_size = benchmark_settings["model_size"][index]
    command = f"/usr/lib/nsight-systems/host-linux-x64/QdstrmImporter -i cs336_systems/nsys_reports/{model_size}.qdstrm -o cs336_systems/nsys_reports/{model_size}.nsys-rep"
    print(command)

for index in range(length):
    model_size = benchmark_settings["model_size"][index]
    command = f"nsys stats --format csv -r nvtxsum cs336_systems/nsys_reports/{model_size}.nsys-rep -o cs336_systems/nsys_reports/{model_size}"
    print(command)

rm cs336_systems/nsys_reports/*.qdstrm cs336_systems/nsys_reports/*.nsys-rep cs336_systems/nsys_reports/*.sqlite cs336_systems/nsys_reports/*.csv
nsys profile -o cs336_systems/nsys_reports/tiny python -m cs336_systems.benchmark --d_model 384 --d_ff 1536 --num_layers 6 --num_heads 6 --mixed_precision True
nsys profile -o cs336_systems/nsys_reports/small python -m cs336_systems.benchmark --d_model 768 --d_ff 3072 --num_layers 12 --num_heads 12 --mixed_precision True
nsys profile -o cs336_systems/nsys_reports/medium python -m cs336_systems.benchmark --d_model 1024 --d_ff 4096 --num_layers 24 --num_heads 16 --mixed_precision True
nsys profile -o cs336_systems/nsys_reports/deep-narrow python -m cs336_systems.benchmark --d_model 512 --d_ff 2048 --num_layers 20 --num_heads 8 --mixed_precision True
nsys profile -o cs336_systems/nsys_reports/ffn-heavy python -m cs336_systems.benchmark --d_model 768 --d_ff 6144 --num_layers 12 --num_heads 12 --mixed_precision True
/usr/lib/nsight-systems/host-li

In [3]:
## GPU time after wrapping forward steps with autocast with bfloat16
## Backward benefit from mixed precision much, where matrix computation is heavily used.
## Medium model shows better running performance with mixed precision, which had d_model=1024, the largest tensor used in our test.

benchmark_settings = {
    "model_size": ["tiny", "small", "medium", "deep-narrow", "ffn-heavy"],
    "d_model": [384, 768, 1024, 512, 768],
    "d_ff": [1536, 3072, 4096, 2048, 6144],
    "num_layers": [6, 12, 24, 20, 12],
    "num_heads": [6, 12, 16, 8, 12]
}

output_dict = {f"{key}_{agg}": list() for key in ["forward", "backward", "optimizer"] for agg in ["avg", "std"]}
for index in range(length):
    model_size = benchmark_settings["model_size"][index]
    result = pd.read_csv(f"./nsys_reports/{model_size}_nvtxsum.csv")
    for key in ["forward", "backward", "optimizer"]:
        output_dict[f"{key}_avg"].append(result[result["Range"] == f"bench/{key}"]["Avg (ns)"].values[0] / 1000000.0)
        output_dict[f"{key}_std"].append(result[result["Range"] == f"bench/{key}"]["StdDev (ns)"].values[0] / 1000000.0)

output_df = pd.DataFrame({**benchmark_settings, **output_dict})
output_df.to_csv("benchmark_timing_nvtx_mixed.csv", index=False)
output_df

,model_size,d_model,d_ff,num_layers,num_heads,forward_avg,forward_std,backward_avg,backward_std,optimizer_avg,optimizer_std
0,tiny,384,1536,6,6,7.862989,3.508394,15.416830,11.283107,14.172635,12.662876
1,small,768,3072,12,12,15.282448,7.467474,44.206742,17.887079,22.980152,16.239375
2,medium,1024,4096,24,16,37.133028,12.091530,120.741688,19.829383,53.291590,31.518826
3,deep-narrow,512,2048,20,8,31.367884,23.463645,79.547446,46.407647,49.607878,41.074102
4,ffn-heavy,768,6144,12,12,17.567263,12.383697,58.344554,24.206555,31.064504,31.585408


### Memory profiling

In [2]:
benchmark_settings = {
    "model_size": ["medium"],
    "d_model": [1024],
    "d_ff": [4096],
    "num_layers": [24],
    "num_heads": [16]
}

params = ["d_model", "d_ff", "num_layers", "num_heads"]
length = len(benchmark_settings["model_size"])
for key in params:
    assert len(benchmark_settings[key]) == length, f"Length mismatch for {key}"

for index in range(length):
    input_dict = {key: benchmark_settings[key][index] for key in params}
    result = benchmark_memory(**input_dict)